### Extract & Load
Extract data from Volume and load into bronze layer


In [0]:
from pyspark.sql.functions import current_timestamp
import os

CATALOG = "northwind_raw_data"
SCHEMA  = "source_sql_db"
CSV_DIR = "/Volumes/northwind_raw_data/source_sql_db/raw_files/northwind_data"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

csv_files = [f for f in os.listdir(CSV_DIR) if f.endswith(".csv")]

for csv_file in csv_files:
    table_name = csv_file.replace(".csv", "")
    file_path = os.path.join(CSV_DIR, csv_file)

    df = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv(file_path) \
        .withColumn("_updated_at", current_timestamp())

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{CATALOG}.{SCHEMA}.{table_name}")

    print(f"  ✅ {table_name} ingested ({df.count()} rows)")

print("Ingestion complete")